In [45]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import rarfile
import tarfile
import py7zr
import pandas as pd
#!pip install thefuzz Levenshtein
from thefuzz import process
import tqdm
from tqdm.auto import tqdm

In [46]:
def move_contents(src_dir, dst_dir, overwrite=True):
    if not src_dir.is_dir():
        raise NotADirectoryError(f"Source must be a directory: {src_dir}")
        
    os.makedirs(dst_dir, exist_ok=True)
    
    for item in src_dir.iterdir():
        src_path = src_dir / item
        dst_path = dst_dir / item
        if src_path.is_file():
            shutil.move(src_path, dst_path)
        elif src_path.is_dir() and dst_path.is_dir():
            move_contents(src_path, dst_path)

    src_dir_size = sum(file.stat().st_size for file in src_dir.rglob('*') if file.is_file())
    src_dir_empty = not any(src_dir.iterdir())
    if src_dir_empty or src_dir_size==0:
        shutil.rmtree(src_dir)

In [47]:
def is_base_zipfile(zip_path, filename='BasicFile.csv'):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        namelist = zf.namelist()
        return filename in namelist

In [48]:
def is_base_rar(rar_path, filename='BasicFile.csv'):
    with rarfile.RarFile(item) as rf:
        file_path_list = rf.namelist()
    file_list = []
    for item in file_path_list:
        file_list.append((Path(item).name))
    return filename in file_list

In [49]:
def base_dir_to_zip(dir_path, output_zip_path):
    csv_files = [item for item in dir_path.iterdir() if item.is_file() and item.suffix.lower() == '.csv']
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for csv_file in csv_files:
            zipf.write(csv_file, arcname=csv_file.name)
    shutil.rmtree(dir_path)

In [50]:
def rar_to_zip(rar_path, output_zip_path):
    extraction_dir = Path.cwd() / "ext_dir"
    with rarfile.RarFile(rar_path) as rf:
        rf.extractall(extraction_dir)
    base_dir_to_zip(extraction_dir, output_zip_path)

In [51]:
def is_base_dir(dir_path, filename='BasicFile.csv'):
    file_list = []
    for item in dir_path.iterdir():
        if item.is_file():
            file_list.append((Path(item).name))
    return filename in file_list

In [52]:
def aggregate_basic_zip(curr_path, parent_dir, irrelevant_files_dir, basic_zip_dir, processed_arch_dir, unprocessed_arch_dir, arch_extn):
    # if subdirectories present at current path, move their content to parent directory recursively
    for item in tqdm(parent_dir.iterdir(), desc="Handling directories..."):
        if item.is_dir() and item.name != "00_new_folder":
            if is_base_dir(item):
                item_name = f"{item.stem}.zip"
                output_zip_path = basic_zip_dir / item_name
                base_dir_to_zip(item, output_zip_path)
                
            else:
                move_contents(item, parent_dir)
        
    # now no directories are present at current location
    for item in tqdm(parent_dir.iterdir(), desc='Extracting archives...'):
        if item.is_file():
            # setting path to move the file after processing
            item_dir = item.parent
            item_name = item.name
            item_stem = item.stem    #item name without extension
            processed_arch = processed_arch_dir / item_name
            basic_zip_dest = basic_zip_dir / item_name
            unprocessed_arch = unprocessed_arch_dir / item_name
            irrelevant_file = irrelevant_files_dir / item_name
            
            if item.suffix in arch_extn:
                new_folder = parent_dir / "00_new_folder"
                os.makedirs(new_folder, exist_ok=True)
    
                try:
                    if zipfile.is_zipfile(item):
                        if is_base_zipfile(item):
                            shutil.move(item, basic_zip_dest)
                        else:
                            with zipfile.ZipFile(item, 'r') as zf:
                                zf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif rarfile.is_rarfile(item):
                        if is_base_rar(item):
                            item_name = f"{item_stem}.zip"
                            output_zip_path = item_dir / item_name
                            rar_to_zip(item, output_zip_path)
                            shutil.move(output_zip_path, basic_zip_dest)
                        else:
                            with rarfile.RarFile(item) as rf:
                                rf.extractall(new_folder)
                            shutil.move(item, processed_arch)
                            
                    elif py7zr.is_7zfile(str(item)):
                        with py7zr.SevenZipFile(item, mode='r') as seven_zip:
                            seven_zip.extractall(path=new_folder)
                        shutil.move(item, processed_arch)
                        
                    elif tarfile.is_tarfile(item):
                        print(item)
                        with tarfile.open(item, 'r') as tar:
                            tar.extractall(new_folder)
                        shutil.move(item, processed_arch)
                        
                    else:
                        shutil.move(item, unprocessed_arch)
                        
                except Exception as e:
                    shutil.move(item, unprocessed_arch)
                    
            else:
                shutil.move(item, irrelevant_file)

    for item in parent_dir.iterdir():
        if item.is_dir() and item.name == "00_new_folder":
            aggregate_basic_zip(curr_path, item, irrelevant_files_dir, basic_zip_dir, processed_arch_dir, unprocessed_arch_dir, arch_extn)

In [53]:
def main():
    curr_path = Path.cwd()
    parent_dir = curr_path / "01_parent_directory"
    irrelevant_files_dir = curr_path / "02_unnecessary_files"
    basic_zip_dir = curr_path / "03_basic_zip_files"
    processed_archives_dir = curr_path / "04_processed_archives"
    unprocessed_archives_dir = curr_path / "05_unprocessed_archives"

    os.makedirs(irrelevant_files_dir, exist_ok=True)
    os.makedirs(basic_zip_dir, exist_ok=True)
    os.makedirs(processed_archives_dir, exist_ok=True)
    os.makedirs(unprocessed_archives_dir, exist_ok=True)

    arch_extn = ['.zip', '.rar', '.tar', '.gz', '.tgz', '.bz2', '.7z']
    aggregate_basic_zip(curr_path, parent_dir, irrelevant_files_dir, basic_zip_dir, processed_archives_dir, unprocessed_archives_dir, arch_extn)

In [54]:
main()

Extracting archives...: 0it [00:00, ?it/s]

Extracting archives...: 0it [00:00, ?it/s]

Extracting archives...: 0it [00:00, ?it/s]

Extracting archives...: 0it [00:00, ?it/s]